# 05. Residual Model Training

This notebook trains the canonical control-anchored residual model. The model predicts a latent perturbation residual conditioned on the control anchor, condition identity, cell identity, and dose. This notebook intentionally excludes cell-aware MMD so that it can serve as a clean reference model in later ablations.


In [8]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [9]:
RUN_NAME = "residual_base"
RUN_DIR = RUNS_DIR / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
for p in [RUN_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

@dataclass
class CFG:
    split_col: str = "split_ho_pathway"
    pca_dim: int = 25
    batch_size: int = 512
    epochs: int = 20
    lr: float = 1e-3
    wd: float = 1e-5
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    alpha_topk: float = 2.0
    dropout: float = 0.1
    eval_topk: tuple = (20, 50)
    num_workers: int = 0
    pin_memory: bool = False
cfg = CFG()
save_json(asdict(cfg), RUN_DIR / "config_resolved.json")


In [10]:
adata, X = load_adata(split_col=cfg.split_col)
obs = pd.read_parquet(PROCESSED_DIR / "datasets" / "adata_obs_processed.parquet")
adata.obs = obs.copy()

X_pca = np.load(PROCESSED_DIR / "pca" / f"X_pca_{cfg.pca_dim}d.npy")
X0_pca = np.load(PROCESSED_DIR / "datasets" / f"X0_pca_{cfg.pca_dim}d.npy")
DELTA_pca = np.load(PROCESSED_DIR / "datasets" / f"DELTA_pca_{cfg.pca_dim}d.npy")

with open(PROCESSED_DIR / "encoders" / "drug_encoder.pkl", "rb") as f:
    drug_enc = pickle.load(f)
with open(PROCESSED_DIR / "encoders" / "cell_encoder.pkl", "rb") as f:
    cell_enc = pickle.load(f)


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [11]:
train_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="train")
test_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="test")
ood_ds = ChemResidualDataset(adata, X_pca, X0_pca, DELTA_pca, split="ood")

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
ood_loader = DataLoader(ood_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

model = ResidualDoseResponseModel(
    latent_dim=cfg.pca_dim,
    n_drugs=len(drug_enc.classes_),
    n_cells=len(cell_enc.classes_),
    emb_dim=cfg.emb_dim,
    hidden=cfg.hidden,
    dose_hidden=cfg.dose_hidden,
    dropout=cfg.dropout,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

history = []
best_score = -1e9
best_path = CKPT_DIR / "best_residual_base.pt"


In [12]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


In [15]:
last_ckpt_path = RUN_DIR / "checkpoints" / "last_checkpoint.pt"
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)

start_epoch = 1
history = []
best_score = -1e18

if last_ckpt_path.exists():
    ckpt = torch.load(last_ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_score = ckpt.get("best_score", -1e18)
    history = ckpt.get("history", [])
    print(f"✅ Resuming from epoch {start_epoch}")
else:
    print("Starting fresh training")

# ----------------------------------
# Training loop
# ----------------------------------
for epoch in range(start_epoch, cfg.epochs + 1):
    model.train()
    train_losses, train_mse, train_topk = [], [], []

    for batch in train_loader:
        x0 = batch["x0"].to(DEVICE)
        delta_true = batch["delta"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose = batch["dose"].to(DEVICE)

        optimizer.zero_grad()
        delta_hat, _ = model(x0, drug_idx, cell_idx, dose)

        loss_mse = F.mse_loss(delta_hat, delta_true)
        mask_topk = topk_mask_from_true(delta_true, k=min(10, cfg.pca_dim))
        loss_topk = ((delta_hat - delta_true) ** 2 * mask_topk).sum() / (mask_topk.sum() + 1e-8)

        loss = loss_mse + cfg.alpha_topk * loss_topk
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_mse.append(loss_mse.item())
        train_topk.append(loss_topk.item())

    test_metrics, test_per_cell, test_group, _, _, _ = evaluate_loader(
        model, test_loader, eval_topk=cfg.eval_topk, split_name="test"
    )
    ood_metrics, ood_per_cell, ood_group, _, _, _ = evaluate_loader(
        model, ood_loader, eval_topk=cfg.eval_topk, split_name="ood"
    )

    score = 0.7 * test_metrics["r2_top50"] + 0.3 * ood_metrics["r2_top50"]

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "train_mse": float(np.mean(train_mse)),
        "train_topk": float(np.mean(train_topk)),
        "test_r2_top50": test_metrics["r2_top50"],
        "ood_r2_top50": ood_metrics["r2_top50"],
        "test_r2_full": test_metrics["r2_full"],
        "ood_r2_full": ood_metrics["r2_full"],
        "selection_score": float(score),
    }
    history.append(row)

    # save history every epoch
    pd.DataFrame(history).to_csv(RUN_DIR / "training_history.csv", index=False)

    # save last checkpoint every epoch
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_score": best_score,
        "history": history,
    }, last_ckpt_path)

    print(
        f"Epoch {epoch:02d} | "
        f"TrainLoss {row['train_loss']:.4f} | "
        f"Test top50 {row['test_r2_top50']:.4f} | "
        f"OOD top50 {row['ood_r2_top50']:.4f}"
    )

    if score > best_score:
        best_score = score

        # save best model weights
        torch.save(model.state_dict(), best_path)

        # save best evaluation artifacts
        test_per_cell.to_csv(RUN_DIR / "best_test_per_cell.csv", index=False)
        ood_per_cell.to_csv(RUN_DIR / "best_ood_per_cell.csv", index=False)
        test_group.to_csv(RUN_DIR / "best_test_groupwise.csv", index=False)
        ood_group.to_csv(RUN_DIR / "best_ood_groupwise.csv", index=False)

        # also save a full best checkpoint
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_score": best_score,
            "history": history,
        }, RUN_DIR / "checkpoints" / "best_checkpoint.pt")

        print("  ✅ saved best model")

Starting fresh training
Epoch 01 | TrainLoss 1.8147 | Test top50 0.5776 | OOD top50 0.5570
  ✅ saved best model
Epoch 02 | TrainLoss 1.8097 | Test top50 0.5780 | OOD top50 0.5550
Epoch 03 | TrainLoss 1.8059 | Test top50 0.5761 | OOD top50 0.5554
Epoch 04 | TrainLoss 1.8037 | Test top50 0.5768 | OOD top50 0.5514
Epoch 05 | TrainLoss 1.8020 | Test top50 0.5764 | OOD top50 0.5504
Epoch 06 | TrainLoss 1.8005 | Test top50 0.5772 | OOD top50 0.5553
Epoch 07 | TrainLoss 1.7996 | Test top50 0.5764 | OOD top50 0.5500
Epoch 08 | TrainLoss 1.7987 | Test top50 0.5769 | OOD top50 0.5490
Epoch 09 | TrainLoss 1.7981 | Test top50 0.5758 | OOD top50 0.5393
Epoch 10 | TrainLoss 1.7971 | Test top50 0.5755 | OOD top50 0.5498
Epoch 11 | TrainLoss 1.7967 | Test top50 0.5769 | OOD top50 0.5517
Epoch 12 | TrainLoss 1.7962 | Test top50 0.5764 | OOD top50 0.5451
Epoch 13 | TrainLoss 1.7957 | Test top50 0.5752 | OOD top50 0.5481
Epoch 14 | TrainLoss 1.7950 | Test top50 0.5758 | OOD top50 0.5460
Epoch 15 | TrainL

In [16]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final_test_metrics, final_test_per_cell, _, pred_test, true_test, x0_test = evaluate_loader(model, test_loader, eval_topk=cfg.eval_topk, split_name="test")
final_ood_metrics, final_ood_per_cell, _, pred_ood, true_ood, x0_ood = evaluate_loader(model, ood_loader, eval_topk=cfg.eval_topk, split_name="ood")

final_overall = pd.DataFrame([
    {"split": "test", "model": "ResidualDoseResponseModel", **final_test_metrics},
    {"split": "ood", "model": "ResidualDoseResponseModel", **final_ood_metrics},
])
final_overall.to_csv(RUN_DIR / "final_overall_metrics.csv", index=False)
pd.concat([final_test_per_cell, final_ood_per_cell], ignore_index=True).to_csv(RUN_DIR / "final_per_cell_metrics.csv", index=False)
np.save(RUN_DIR / "pred_test.npy", pred_test)
np.save(RUN_DIR / "pred_ood.npy", pred_ood)

final_overall


,split,model,r2_full,pearson_rowmean,mse,collapse_ratio,mean_shift_error,r2_top20,r2_top50
0,test,ResidualDoseResponseModel,0.636585,0.784522,0.340675,0.640241,0.441033,0.499407,0.577561
1,ood,ResidualDoseResponseModel,0.614290,0.772493,0.359441,0.616443,0.454386,0.484087,0.557015


This run provides the simpler reference model. The next notebook introduces cell-aware MMD, which corresponds to the stronger v2 logic already present in the current pipeline.
